In [84]:
# import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
# bike sharing demand dataset
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/bikeshare.csv"
df = pd.read_csv(url)

df.shape

(10886, 12)

In [98]:
df.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,target,hour,day_of_week,is_weekend,temp_humidity_interaction,windspeed_temp_ratio,log_count,temp_binned
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,0,0,5,1,797.04,0.0,2.833213,1
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,0,1,5,1,721.60,0.0,3.713572,1
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,0,2,5,1,721.60,0.0,3.496508,1
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,0,3,5,1,738.00,0.0,2.639057,1
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,0,4,5,1,738.00,0.0,0.693147,1


In [86]:
# info and parse datetime
df['datetime'] = pd.to_datetime(df['datetime'])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    10886 non-null  datetime64[ns]
 1   season      10886 non-null  int64         
 2   holiday     10886 non-null  int64         
 3   workingday  10886 non-null  int64         
 4   weather     10886 non-null  int64         
 5   temp        10886 non-null  float64       
 6   atemp       10886 non-null  float64       
 7   humidity    10886 non-null  int64         
 8   windspeed   10886 non-null  float64       
 9   casual      10886 non-null  int64         
 10  registered  10886 non-null  int64         
 11  count       10886 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(8)
memory usage: 1020.7 KB


In [87]:
# baseline preprocessing, high demand =1 and low demand =0
df['target'] = (df['count'] > df['count'].median()).astype(int)

baseline_features = ['temp', 'atemp', 'humidity', 'windspeed']

X = df[baseline_features]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [88]:
# baseline model training and ecaluation
baseline_model = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred)
print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.6864095500459136


In [89]:
# feature extraction from datetime feature
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

In [90]:
# interaction features
df['temp_humidity_interaction'] = df['temp'] * df['humidity']
df['windspeed_temp_ratio'] = df['windspeed'] / (df['temp'] + 1e-5)

In [91]:
# log transform skewed features to reduce the impact of outliers
df['log_count'] = np.log1p(df['count'])

In [92]:
# binning temperature into 5 equal-width bins
df['temp_binned'] = pd.cut(df['temp'], bins=5, labels=[0,1,2,3,4])
df['temp_binned'] = df['temp_binned'].astype(int)

In [93]:
# engineered features list
engineered_features = [
    'temp', 'atemp', 'humidity', 'windspeed',
    'hour', 'day_of_week', 'is_weekend',
    'temp_humidity_interaction',
    'windspeed_temp_ratio',
    'temp_binned'
]

X2 = df[engineered_features]
y2 = df['target']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

In [94]:
model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X_train2, y_train2)

y_pred2 = model2.predict(X_test2)

engineered_accuracy = accuracy_score(y_test2, y_pred2)
print("Engineered Model Accuracy:", engineered_accuracy)

Engineered Model Accuracy: 0.8783287419651056


In [95]:
# accuracy comparision
print("Baseline Accuracy   :", baseline_accuracy)
print("Engineered Accuracy :", engineered_accuracy)

print("Improvement (Delta) :", engineered_accuracy - baseline_accuracy)

Baseline Accuracy   : 0.6864095500459136
Engineered Accuracy : 0.8783287419651056
Improvement (Delta) : 0.19191919191919193


Why each feature helps:
- Hour → captures rush hours (morning/evening demand spikes)
- Day-of-week → weekdays vs weekends demand patterns
- Is_weekend → leisure travel increases on weekends
- Temp × Humidity → combined weather comfort factor affects bike usage
- Wind speed / Temp ratio → harsh weather impact
- Log(count) → reduces skewness for better learning patterns
- Temp binning → converts continuous temperature into meaningful categories (cold/mild/hot)